# YOLOE-26 · VisDrone sliced fine-tuning with a test-overlap sweep

Run this notebook **after** `00_data_and_protocol_SageMaker_VisDrone.ipynb`, in the same persistent SageMaker JupyterLab working directory. It uses the finalized ClearML protocol Dataset and the same local VisDrone files as the original sliced-training notebook.

This notebook is the YOLO-World test-overlap sweep with **only the model changed to YOLOE-26s**. Zero-shot uses the released `yoloe-26s-seg.pt` weights (only its boxes are evaluated). Fine-tuning builds the plain-detection architecture `yoloe-26s.yaml`, loads those released weights, and trains with Ultralytics' text-prompt trainer `YOLOETrainerFromScratch`. Despite its name it starts from the released weights, so this is fine-tuning; it keeps the classification head unfused, so the fine-tuned model still accepts all 10 prompts, like YOLO-World's `WorldTrainer`. `YOLOEPETrainer` is NOT used: it fuses the 7 base-class embeddings into the head, after which novel prompts can no longer be set. Outputs go to `weights/yoloe_26/`, `results/yoloe_26/` and a new ClearML project. Every other model/evaluation parameter is identical to the YOLO-World run. Training images retain the original fixed `20%` horizontal and vertical tile overlap. The **only experimental change** is at sliced evaluation time: the same zero-shot and fine-tuned checkpoints are evaluated with test overlaps of `10%`, `20%`, `30%`, `40%`, and `50%`. Whole-image inference is run once per checkpoint as a reference. All overlap conditions use the same evaluation images, prompts, tile size, thresholds, NMS settings and weights.

For a quick pipeline check set `run_mode="smoke"`; smoke results are not paper results. For final results use `run_mode="full"`. The output tables include AP, AP50, base/novel AP, size-bin AP, operating-point precision/recall/F1, latency and average tile count for every test overlap.


In [1]:
# Restart the kernel after the first installation if prompted, then run all cells top-to-bottom.
%pip install -q "ultralytics>=8.4,<9" "clearml>=1.16,<3" "pycocotools>=2.0.7,<3" "pillow>=10,<13" "numpy>=1.24,<3" "pandas>=2,<3" "tqdm>=4.66,<5"


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from getpass import getpass

os.environ["CLEARML_WEB_HOST"] = "https://app.clear.ml"
os.environ["CLEARML_API_HOST"] = "https://api.clear.ml"
os.environ["CLEARML_FILES_HOST"] = "https://files.clear.ml"

os.environ["CLEARML_API_ACCESS_KEY"] = getpass("ClearML access key: ")
os.environ["CLEARML_API_SECRET_KEY"] = getpass("ClearML secret key: ")

print("ClearML credentials set for this kernel.")

ClearML access key:  ········
ClearML secret key:  ········


ClearML credentials set for this kernel.


In [3]:
import csv
import hashlib
import json
import os
import platform
import random
import shutil
import time
import uuid
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from tqdm.auto import tqdm

@dataclass
class Config:
    # The project root is the SAME current directory used by notebook 00.
    project_root: str = "SageMaker_VisDrone_OVD_2026"
    project_name: str = "UAV-OVD-Research"
    task_name_prefix: str = "Experiment 1 final test yoloe"
    clearml_project: str = "SageMaker-Aerial-OVD-2026/YOLOE-26-VisDrone-FT-Whole-Sliced-2026"   # YOLOE
    yolo_checkpoint: str = "yoloe-26s-seg.pt"      # YOLOE: released weights
    yolo_yaml: str = "yoloe-26s.yaml"              # YOLOE: plain-detection architecture
    grounding_dino_checkpoint: str = "IDEA-Research/grounding-dino-tiny"

    run_mode: str = "full"                  # "smoke" or "full".
    eval_split: str = "val"                  # Switch to "test" only after freezing ALL choices.
    smoke_images: int = 50
    epochs: int = 80
    smoke_epochs: int = 1
    batch: int = 8
    yolo_image_size: int = 1280
    yolo_extraction_threshold: float = 0.001
    yolo_local_nms_iou: float = 0.70
    gdino_box_threshold: float = 0.01
    gdino_text_threshold: float = 0.20
    gdino_shortest_edge: int = 800
    gdino_longest_edge: int = 1333
    operating_threshold_yolo: float = 0.25
    operating_threshold_gdino: float = 0.25
    slice_height: int = 640
    slice_width: int = 640
    # These two values are used only when generating TRAINING tiles and remain fixed.
    overlap_height_ratio: float = 0.20
    overlap_width_ratio: float = 0.20
    # Only sliced TEST inference is swept. The model is not retrained for each value.
    test_overlap_ratios: tuple = (0.10, 0.20, 0.30, 0.40, 0.50)
    include_whole_prediction_in_sliced: bool = False
    global_nms_iou: float = 0.50
    max_detections_per_image: int = 500
    evaluation_iou: float = 0.50
    seed: int = 42
    save_to_google_drive: bool = False
    drive_output_folder: str = "MyDrive/UAV_OVD_Research/Experiment_1"
    skip_completed_conditions: bool = True
    register_manifest_as_clearml_dataset: bool = True
    upload_full_raw_dataset_to_clearml: bool = False
    verify_image_hashes: bool = True

CFG = Config()
assert CFG.run_mode in {"smoke", "full"}
assert CFG.eval_split in {"val", "test"}
assert 0 <= CFG.yolo_extraction_threshold <= CFG.operating_threshold_yolo < 1
assert 0 <= CFG.overlap_height_ratio < 1 and 0 <= CFG.overlap_width_ratio < 1
assert tuple(CFG.test_overlap_ratios) == (0.10, 0.20, 0.30, 0.40, 0.50)
assert all(0 <= ratio < 1 for ratio in CFG.test_overlap_ratios)
assert CFG.slice_height >= 64 and CFG.slice_width >= 64
assert 0 <= CFG.yolo_local_nms_iou <= 1
assert 0 <= CFG.global_nms_iou <= 1
assert 0 <= CFG.evaluation_iou <= 1
if CFG.eval_split == "test" and CFG.run_mode != "full":
    raise ValueError("Reserved test-dev evaluation must use the full protocol, never a smoke subset.")
if not torch.cuda.is_available():
    raise RuntimeError("Use a SageMaker GPU kernel for this train-and-evaluate notebook.")

random.seed(CFG.seed)
np.random.seed(CFG.seed)
torch.manual_seed(CFG.seed)
torch.cuda.manual_seed_all(CFG.seed)
ROOT = (Path.cwd().resolve() / CFG.project_root)
DATA_ROOT = ROOT / "visdrone_det_original"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
MODEL_FAMILY = "yoloe_26"   # Folder name under runs/, weights/ and results/.
RUN_DIR = ROOT / "runs" / MODEL_FAMILY / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
WEIGHTS_DIR = ROOT / "weights" / MODEL_FAMILY / RUN_ID
RESULTS_DIR = ROOT / "results" / MODEL_FAMILY / RUN_ID
WEIGHTS_DIR.mkdir(parents=True, exist_ok=False)
RESULTS_DIR.mkdir(parents=True, exist_ok=False)
YOLO_CONFIG_DIR = RUN_DIR / "ultralytics_config"
YOLO_CONFIG_DIR.mkdir()
os.environ["YOLO_CONFIG_DIR"] = str(YOLO_CONFIG_DIR)
print("Run:", RUN_ID, "| Device:", torch.cuda.get_device_name(0))
print("Local training workspace:", RUN_DIR)
print("Persistent local weights:", WEIGHTS_DIR)
print("Persistent local results:", RESULTS_DIR)
print("Free space (GiB):", round(shutil.disk_usage(ROOT).free / 1024**3, 2))
print("New ClearML project:", CFG.clearml_project)
print("Fixed training overlap:", CFG.overlap_height_ratio, CFG.overlap_width_ratio)
print("Test overlap sweep:", CFG.test_overlap_ratios)


Run: 20261028T153524Z_5dd65a8e | Device: Tesla T4
Local training workspace: /home/sagemaker-user/SageMaker_VisDrone_OVD_2026/runs/yoloe_26/20261028T153524Z_5dd65a8e
Persistent local weights: /home/sagemaker-user/SageMaker_VisDrone_OVD_2026/weights/yoloe_26/20261028T153524Z_5dd65a8e
Persistent local results: /home/sagemaker-user/SageMaker_VisDrone_OVD_2026/results/yoloe_26/20261028T153524Z_5dd65a8e
Free space (GiB): 8.98
New ClearML project: SageMaker-Aerial-OVD-2026/YOLOE-26-VisDrone-FT-Whole-Sliced-2026
Fixed training overlap: 0.2 0.2
Test overlap sweep: (0.1, 0.2, 0.3, 0.4, 0.5)


## 1 · Load the *published* protocol, not a new random split

Do not modify the v1 Dataset. The validation images with annotated novel targets are omitted **only during training model selection**; the evaluation below still includes all original ground-truth categories. The local SHA-256 checks make accidental substitutions or edits visible. Never train on val/test-dev.


In [4]:
from clearml import Dataset, Task

id_file = ROOT / "clearml_protocol_dataset_id.txt"
if not id_file.is_file():
    raise FileNotFoundError(f"Run notebook 00 first; missing {id_file}")
PROTOCOL_ID = id_file.read_text(encoding="utf-8").strip()
ds = Dataset.get(dataset_id=PROTOCOL_ID)
if not ds.is_final():
    raise RuntimeError("Protocol Dataset is not finalized. Do not train on mutable splits.")
PROTOCOL_DIR = Path(ds.get_local_copy()).resolve()

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

required = {"categories_and_prompts.json", "protocol.json", "summary.json",
            "train.csv", "val.csv", "test.csv", "train_base_safe.csv", "protocol_fingerprint.txt"}
if not required.issubset({p.name for p in PROTOCOL_DIR.iterdir() if p.is_file()}):
    raise RuntimeError("ClearML protocol Dataset is missing a required v1 file.")
digest = hashlib.sha256()
for name in sorted(required - {"protocol_fingerprint.txt"}):
    content = (PROTOCOL_DIR / name).read_bytes()
    digest.update(name.encode() + b"\x00" + content + b"\x00")
FINGERPRINT = digest.hexdigest()
if FINGERPRINT != (PROTOCOL_DIR / "protocol_fingerprint.txt").read_text().strip():
    raise RuntimeError("Protocol contents do not match their published fingerprint.")
meta_fingerprint = (ds.get_metadata() or {}).get("protocol_fingerprint")
if meta_fingerprint != FINGERPRINT:
    raise RuntimeError("ClearML Dataset metadata fingerprint does not match its files.")
spec = json.loads((PROTOCOL_DIR / "protocol.json").read_text(encoding="utf-8"))
cats = json.loads((PROTOCOL_DIR / "categories_and_prompts.json").read_text(encoding="utf-8"))
if spec["version"] != "1.0.0":
    raise RuntimeError("This notebook expects the published protocol v1.0.0.")
CATEGORY_IDS = sorted(int(k) for k in cats["official_category_id_to_name"])
CATEGORY_NAMES = [cats["official_category_id_to_name"][str(k)] for k in CATEGORY_IDS]
PROMPTS = [cats["prompt_by_category_name"][name] for name in CATEGORY_NAMES]
BASE_NAMES = cats["base_category_names"]
NOVEL_NAMES = cats["novel_category_names"]
BASE_IDS = set(cats["base_official_ids"])
NOVEL_IDS = set(cats["novel_official_ids"])
assert len(CATEGORY_IDS) == 10 and set(CATEGORY_IDS) == BASE_IDS | NOVEL_IDS
assert BASE_IDS.isdisjoint(NOVEL_IDS) and len(set(PROMPTS)) == 10
BASE_ID_TO_INDEX = {cat: i for i, cat in enumerate(sorted(BASE_IDS))}

def load_csv(name):
    with (PROTOCOL_DIR / name).open(newline="", encoding="utf-8") as stream:
        return list(csv.DictReader(stream))

train_rows = load_csv("train.csv")
val_rows = load_csv("val.csv")
test_rows = load_csv("test.csv")
safe_train_rows = load_csv("train_base_safe.csv")
assert len(train_rows) == spec["expected_images"]["train"]
assert len(val_rows) == spec["expected_images"]["val"]
assert len(test_rows) == spec["expected_images"]["test"]
train_keys = {r["image_relpath"] for r in train_rows}
assert train_keys.isdisjoint({r["image_relpath"] for r in val_rows})
assert train_keys.isdisjoint({r["image_relpath"] for r in test_rows})
assert {r["image_relpath"] for r in val_rows}.isdisjoint({r["image_relpath"] for r in test_rows})
assert {r["image_relpath"] for r in safe_train_rows}.issubset(train_keys)
assert all(int(r["num_novel_objects"]) == 0 for r in safe_train_rows)
safe_val_rows = [r for r in val_rows if int(r["num_novel_objects"]) == 0]
assert safe_val_rows, "No novel-free val images available for training validation."
eval_rows_full = val_rows if CFG.eval_split == "val" else test_rows

def chosen(rows, count):
    return sorted(rows, key=lambda r: r["image_relpath"]) if CFG.run_mode == "full" else sorted(
        random.Random(CFG.seed).sample(rows, min(count, len(rows))), key=lambda r: r["image_relpath"]
    )

train_selection = chosen(safe_train_rows, CFG.smoke_images)
val_selection = chosen(safe_val_rows, CFG.smoke_images)
eval_selection = chosen(eval_rows_full, CFG.smoke_images)

def local_file(relpath):
    path = (DATA_ROOT / relpath).resolve()
    if not path.is_relative_to(DATA_ROOT.resolve()):
        raise RuntimeError(f"Untrusted protocol relative path: {relpath}")
    return path

all_selected = {r["image_relpath"]: r for r in train_selection + val_selection + eval_selection}
for row in tqdm(all_selected.values(), desc="Check frozen image/annotation manifests"):
    for key, sha in (("image_relpath", "image_sha256"), ("annotation_relpath", "annotation_sha256")):
        path = local_file(row[key])
        if not path.is_file():
            raise FileNotFoundError(f"Missing source data: {path}")
        if CFG.verify_image_hashes and sha256_file(path) != row[sha]:
            raise RuntimeError(f"Source data modified since protocol registration: {path}")
print("Frozen Dataset ID:", PROTOCOL_ID, "| fingerprint:", FINGERPRINT)
print("Training/early-stopping/evaluation images:",
      len(train_selection), len(val_selection), len(eval_selection))
print("Base:", BASE_NAMES, "| Novel:", NOVEL_NAMES)
print("Full results?", CFG.run_mode == "full")


Check frozen image/annotation manifests:   0%|          | 0/3243 [00:00<?, ?it/s]

Frozen Dataset ID: 738dce9e1a5440dab34195418a997a20 | fingerprint: 158cb2c904016fa324c236b998393b1a94c176a4eabedb159c76bdeffc66c195
Training/early-stopping/evaluation images: 2695 119 548
Base: ['pedestrian', 'people', 'bicycle', 'car', 'van', 'truck', 'motor'] | Novel: ['tricycle', 'awning-tricycle', 'bus']
Full results? True


## 2 · Create mixed whole-image and sliced training views

This section is unchanged from the original sliced-training experiment. Whole training images are retained and positive `640 × 640` training tiles are generated with the fixed `20%` training overlap. The later test-overlap sweep does not alter this training dataset.


In [5]:
def annotations_for_row(row):
    path = local_file(row["annotation_relpath"])
    records = []
    for lineno, raw in enumerate(path.read_text(encoding="utf-8-sig").splitlines(), 1):
        if not raw.strip():
            continue
        fields = [x.strip() for x in raw.split(",")]
        if len(fields) < 8:
            raise ValueError(f"{path}:{lineno}: expected 8 VisDrone columns")
        x, y, w, h = map(float, fields[:4])
        score, cid = int(fields[4]), int(fields[5])
        truncation, occlusion = int(fields[6]), int(fields[7])
        if cid not in {0, 11, *CATEGORY_IDS}:
            raise ValueError(f"{path}:{lineno}: unexpected category {cid}")
        if w <= 0 or h <= 0:
            continue
        records.append(dict(bbox=[x, y, w, h], score=score, category_id=cid,
                            truncation=truncation, occlusion=occlusion))
    return records


def training_sliding_positions(full, tile, overlap):
    if full <= tile:
        return [0]
    stride = max(1, round(tile * (1 - overlap)))
    points = list(range(0, full - tile + 1, stride))
    if points[-1] != full - tile:
        points.append(full - tile)
    return sorted(set(points))


def yolo_label_line(cid, x1, y1, x2, y2, width, height):
    return (f"{BASE_ID_TO_INDEX[cid]} {(x1+x2)/(2*width):.8f} "
            f"{(y1+y2)/(2*height):.8f} {(x2-x1)/width:.8f} {(y2-y1)/height:.8f}\n")


# This is an annotation-integrity rule, not a changed model hyperparameter.
# Whole images are retained, so large objects remain available even when a
# boundary tile is rejected.
MIN_TILE_VISIBLE_FRACTION = 0.50

TRAIN_DATA_DIR = RUN_DIR / "train_data"
tile_count = 0
whole_count = 0
rejected_boundary_tiles = 0

for split, rows in (("train", train_selection), ("val", val_selection)):
    image_out = TRAIN_DATA_DIR / split / "images"
    label_out = TRAIN_DATA_DIR / split / "labels"
    image_out.mkdir(parents=True)
    label_out.mkdir(parents=True)
    seen = set()

    for row in tqdm(rows, desc=f"Write {split} YOLO labels"):
        source = local_file(row["image_relpath"])
        if source.name in seen:
            raise RuntimeError(f"Duplicate image filename in selected {split} set: {source.name}")
        seen.add(source.name)
        annotations = annotations_for_row(row)

        with Image.open(source) as opened:
            image = opened.convert("RGB")
            W, H = image.size

            # Preserve the original whole-image view exactly as before.
            whole_target = image_out / source.name
            whole_target.symlink_to(source)
            whole_lines = []
            for ann in annotations:
                cid = ann["category_id"]
                if cid not in BASE_IDS:
                    if cid in NOVEL_IDS:
                        raise RuntimeError("A novel-class GT was found in a base-safe training/validation image.")
                    continue
                x, y, w, h = ann["bbox"]
                x1, y1 = max(0.0, min(float(W), x)), max(0.0, min(float(H), y))
                x2, y2 = max(0.0, min(float(W), x + w)), max(0.0, min(float(H), y + h))
                if x2 <= x1 or y2 <= y1:
                    continue
                whole_lines.append(yolo_label_line(cid, x1, y1, x2, y2, W, H))
            (label_out / f"{source.stem}.txt").write_text("".join(whole_lines), encoding="utf-8")
            whole_count += 1

            # The original validation view is deliberately unchanged. Only
            # the training split receives additional sliced examples.
            if split != "train":
                continue

            x_positions = training_sliding_positions(W, CFG.slice_width, CFG.overlap_width_ratio)
            y_positions = training_sliding_positions(H, CFG.slice_height, CFG.overlap_height_ratio)

            for top in y_positions:
                for left in x_positions:
                    right = min(W, left + CFG.slice_width)
                    bottom = min(H, top + CFG.slice_height)
                    tile_w, tile_h = right - left, bottom - top
                    tile_lines = []
                    reject_tile = False

                    for ann in annotations:
                        cid = ann["category_id"]
                        if cid not in BASE_IDS:
                            continue
                        x, y, w, h = ann["bbox"]
                        box_x1 = max(0.0, min(float(W), x))
                        box_y1 = max(0.0, min(float(H), y))
                        box_x2 = max(0.0, min(float(W), x + w))
                        box_y2 = max(0.0, min(float(H), y + h))
                        box_area = max(0.0, box_x2 - box_x1) * max(0.0, box_y2 - box_y1)
                        if box_area <= 0:
                            continue

                        inter_x1 = max(box_x1, float(left))
                        inter_y1 = max(box_y1, float(top))
                        inter_x2 = min(box_x2, float(right))
                        inter_y2 = min(box_y2, float(bottom))
                        inter_area = max(0.0, inter_x2 - inter_x1) * max(0.0, inter_y2 - inter_y1)
                        if inter_area <= 0:
                            continue

                        visible_fraction = inter_area / box_area
                        if visible_fraction < MIN_TILE_VISIBLE_FRACTION:
                            reject_tile = True
                            break

                        tile_lines.append(yolo_label_line(
                            cid,
                            inter_x1 - left, inter_y1 - top,
                            inter_x2 - left, inter_y2 - top,
                            tile_w, tile_h,
                        ))

                    # Whole images already provide background context. Added
                    # slices are restricted to positive, non-severely-cut views
                    # so empty tiles do not overwhelm the original dataset.
                    if reject_tile:
                        rejected_boundary_tiles += 1
                        continue
                    if not tile_lines:
                        continue

                    tile_stem = f"tile__{source.stem}__y{top:05d}_x{left:05d}"
                    tile_path = image_out / f"{tile_stem}.jpg"
                    image.crop((left, top, right, bottom)).save(
                        tile_path, format="JPEG", quality=95, subsampling=0
                    )
                    (label_out / f"{tile_stem}.txt").write_text(
                        "".join(tile_lines), encoding="utf-8"
                    )
                    tile_count += 1


# Base names in ascending official-ID order: model classes 0..6 during training.
train_yaml = RUN_DIR / "base_only.yaml"
train_yaml.write_text(
    f"path: {TRAIN_DATA_DIR.as_posix()}\ntrain: train/images\nval: val/images\n"
    + "names:\n" + "".join(f"  {i}: {cats['official_category_id_to_name'][str(cid)]}\n"
                              for cid, i in BASE_ID_TO_INDEX.items()),
    encoding="utf-8",
)
print("YOLO training YAML:", train_yaml)
print("Whole training/validation images retained:", whole_count)
print("Additional positive training tiles:", tile_count)
print("Rejected boundary tiles:", rejected_boundary_tiles)


Write train YOLO labels:   0%|          | 0/2695 [00:00<?, ?it/s]

Write val YOLO labels:   0%|          | 0/119 [00:00<?, ?it/s]

YOLO training YAML: /home/sagemaker-user/SageMaker_VisDrone_OVD_2026/runs/yoloe_26/20261028T153524Z_5dd65a8e/base_only.yaml
Whole training/validation images retained: 2814
Additional positive training tiles: 6985
Rejected boundary tiles: 9428


## 3 · Create a NEW ClearML experiment project and local output directories

No bucket, AWS credentials, or S3 configuration is needed. This notebook stores the large weights and complete prediction files under the 50 GB SageMaker space. Each run gets a unique ID. Ultralytics automatic ClearML checkpoint upload is disabled to avoid consuming the ClearML file quota. Do not execute this notebook with an already-active unrelated ClearML task. Monitor free disk space when running more models; `!df -h ~` shows the available space.


In [6]:
from ultralytics import YOLOE, __version__ as ULTRALYTICS_VERSION, settings
# Text-prompt trainer: keeps the head unfused so all 10 prompts can be set after fine-tuning.
from ultralytics.models.yolo.yoloe import YOLOETrainerFromScratch

if tuple(int(x) for x in ULTRALYTICS_VERSION.split(".")[:2]) < (8, 4):
    raise RuntimeError(f"YOLOE-26 needs ultralytics>=8.4; found {ULTRALYTICS_VERSION}. Re-run the pip cell and restart.")

# Set YOLO_CONFIG_DIR *before* importing ultralytics. Disabling its auto-ClearML
# callback prevents duplicate tasks and uncontrolled checkpoint uploads. This
# per-run settings directory does not modify other notebooks' YOLO settings.
settings.update({"clearml": False})
if settings["clearml"]:
    raise RuntimeError("Disable Ultralytics ClearML auto-upload before training.")
if Task.current_task() is not None:
    raise RuntimeError("An unrelated ClearML task is active. Restart the kernel before this notebook.")

task = Task.init(
    project_name=CFG.clearml_project,
    task_name=f"{CFG.task_name_prefix} | YOLOE-26 test-overlap sweep | {CFG.run_mode} | {CFG.eval_split} | {RUN_ID}",
    reuse_last_task_id=False,
    auto_connect_frameworks=False,
)
logger = task.get_logger()
cfg_logged = asdict(CFG)
task.connect(cfg_logged, name="experiment_config")
task.connect({"protocol_dataset_id": PROTOCOL_ID, "protocol_fingerprint": FINGERPRINT,
              "local_weights_dir": str(WEIGHTS_DIR),
              "local_results_dir": str(RESULTS_DIR),
              "run_id": RUN_ID, "ultralytics_version": ULTRALYTICS_VERSION,
              "torch_version": torch.__version__, "python": platform.python_version()}, name="provenance")
print("ClearML task:", task.id)

def record_local_file(path):
    path = Path(path).resolve()
    if not path.is_file() or not (path.is_relative_to(WEIGHTS_DIR) or path.is_relative_to(RESULTS_DIR)):
        raise ValueError(f"Expected a stored file under weights/ or results/: {path}")
    return {"local_path": str(path), "file_uri": path.as_uri(),
            "bytes": path.stat().st_size, "sha256": sha256_file(path)}

def store_local_file(source_path, section, filename):
    source_path = Path(source_path).resolve()
    if not source_path.is_file() or section not in {"weights", "results"}:
        raise ValueError("Only existing source files and weights/results sections are allowed.")
    if Path(filename).name != filename or filename in {"", ".", ".."}:
        raise ValueError("Use a plain filename with no path components.")
    destination = (WEIGHTS_DIR if section == "weights" else RESULTS_DIR) / filename
    if destination.exists():
        if not destination.samefile(source_path):
            raise FileExistsError(f"Refusing to overwrite an earlier output: {destination}")
    else:
        # Hard-link where both paths share the EBS volume: avoids using disk twice.
        # If the source is on another filesystem, copy instead.
        temporary = destination.with_name(f".{filename}.{uuid.uuid4().hex}.tmp")
        try:
            try:
                os.link(source_path, temporary)
            except OSError:
                shutil.copy2(source_path, temporary)
            if sha256_file(temporary) != sha256_file(source_path):
                raise RuntimeError(f"Local copy did not pass SHA-256 verification: {source_path}")
            os.replace(temporary, destination)
        finally:
            temporary.unlink(missing_ok=True)
    return record_local_file(destination)

LOCAL_INDEX = {"weights": {}, "results": {}}


WARNING ⚠️ torchvision==0.24 is incompatible with torch==2.8.
Run 'pip install torchvision==0.23' to fix torchvision or 'pip install -U torch torchvision' to update both.
For a full compatibility table see https://github.com/pytorch/vision#installation
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/home/sagemaker-user/SageMaker_VisDrone_OVD_2026/runs/yoloe_26/20261028T153524Z_5dd65a8e/ultralytics_config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
ClearML Task: created new task id=eed748ae19974a7d90c24e513128e52f
ClearML results page: https://app.clear.ml/projects/f957e634b3ab4a9d876c2b8513f94e04/tasks/eed748ae19974a7d90c24e513128e52f/output/log
ClearML task: eed748ae19974a7d90c24e513128e52f
ClearML Monitor: Could not detect iteration reporting, falling back to iterations as seconds-from-start


## 4 · Fine-tune on mixed whole and sliced base-class views

YOLOE: the fine-tuned model is `YOLOE("yoloe-26s.yaml").load("yoloe-26s-seg.pt")` trained with `trainer=YOLOETrainerFromScratch` (text-prompt fine-tuning from the released weights; the head stays unfused, so all 10 prompts can be set afterwards). Train on the original base-safe whole images plus the additional positive training tiles generated above; validate only on the unchanged base-safe whole-image val view, never on test-dev. `best.pt` is selected by the same base-class validation performance as before; `last.pt` is also retained for possible continuation. Both are saved under the **shared local weights folder** immediately after training. The original pretrained checkpoint is also saved for zero-shot reproducibility. We explicitly call `set_classes` with **all ten frozen prompts** after loading each checkpoint; if the fine-tuned checkpoint cannot dynamically accept novel prompts, stop and report this failure rather than claiming novel AP.


In [ ]:
def set_frozen_prompts(model):
    # YOLOE: text prompts are passed together with their MobileCLIP text embeddings.
    model.set_classes(PROMPTS, model.get_text_pe(PROMPTS))
    active = [model.names[k] for k in sorted(model.names)]
    if active[:len(PROMPTS)] != PROMPTS:
        raise RuntimeError(f"Prompt order mismatch: expected {PROMPTS!r}, got {active!r}")
    return model

baseline = YOLOE(CFG.yolo_checkpoint)
pretrained_path = Path(getattr(baseline, "ckpt_path", CFG.yolo_checkpoint))
if not pretrained_path.is_file():
    pretrained_path = Path(CFG.yolo_checkpoint)
if not pretrained_path.is_file():
    raise FileNotFoundError("Ultralytics pretrained checkpoint not found after model loading.")
LOCAL_INDEX["weights"]["pretrained.pt"] = store_local_file(pretrained_path, "weights", "pretrained.pt")
baseline = set_frozen_prompts(baseline)

# Create a fresh model for fine-tuning; no zero-shot prompt changes bleed into training.
# YOLOE: plain-detection architecture initialised from the released weights.
trainer = YOLOE(CFG.yolo_yaml).load(CFG.yolo_checkpoint)

# YOLOE fix: the training loss is created on the first batch from the head's current class count, which an
# earlier warm-up pass leaves at 80. Each training batch carries 7 base-class text prompts, so the loss must use
# 7 classes (otherwise: 'Target size [.., 80] must be the same as input size [.., 7]'). Resetting the head count
# right before the first epoch makes the loss build with 7. This does not change the model or the data.
TRAIN_NC = len(BASE_ID_TO_INDEX)  # 7 base classes = 7 training text prompts

def align_yoloe_head_for_training(yolo_trainer):
    net = getattr(yolo_trainer.model, "module", yolo_trainer.model)  # unwrap DDP if present
    head = net.model[-1]
    criterion = getattr(net, "criterion", None)
    built_nc = getattr(getattr(criterion, "one2many", criterion), "nc", None) if criterion is not None else None
    if built_nc != TRAIN_NC:
        head.nc = TRAIN_NC
        head.no = TRAIN_NC + head.reg_max * 4
        net.criterion = None  # rebuilt on the first batch, on the GPU, with 7 classes

trainer.add_callback("on_train_epoch_start", align_yoloe_head_for_training)
training_results = trainer.train(
    # YOLOE text-prompt trainer takes the same YAML wrapped as {split: {'yolo_data': [yaml]}}.
    data={"train": {"yolo_data": [str(train_yaml)]}, "val": {"yolo_data": [str(train_yaml)]}},
    epochs=CFG.smoke_epochs if CFG.run_mode == "smoke" else CFG.epochs,
    imgsz=CFG.yolo_image_size,
    batch=CFG.batch,
    device=0,
    seed=CFG.seed,
    deterministic=True,
    project=str(RUN_DIR / "training"),
    name="base_safe",
    exist_ok=False,
    workers=4,
    save=True,
    plots=False,
    trainer=YOLOETrainerFromScratch,   # YOLOE: text-prompt fine-tuning, head stays unfused
)
checkpoint_dir = Path(trainer.trainer.save_dir) / "weights"
best_path = checkpoint_dir / "best.pt"
last_path = checkpoint_dir / "last.pt"
for name, path in (("best.pt", best_path), ("last.pt", last_path)):
    if not path.is_file():
        raise FileNotFoundError(f"Training did not save expected checkpoint: {path}")
    LOCAL_INDEX["weights"][name] = store_local_file(path, "weights", name)
print("Checkpoints verified locally:", LOCAL_INDEX["weights"])

# Register local file URIs as metadata, without uploading model bytes to ClearML.
# They are accessible only from an environment with this SageMaker directory mounted.
from clearml import OutputModel
for checkpoint_name in ("best.pt", "last.pt"):
    registered = OutputModel(task=task, name=f"YOLOE-26s {checkpoint_name} | {RUN_ID}",
                             framework="PyTorch", label_enumeration={name: i for i, name in enumerate(BASE_NAMES)},
                             comment=f"VisDrone base-safe protocol {PROTOCOL_ID}; see local checkpoint and frozen prompts")
    registered.update_weights(register_uri=LOCAL_INDEX["weights"][checkpoint_name]["file_uri"],
                              async_enable=False)
finetuned = set_frozen_prompts(YOLOE(str(best_path)))

training_csv = Path(trainer.trainer.save_dir) / "results.csv"
if training_csv.is_file():
    LOCAL_INDEX["results"]["training_history.csv"] = store_local_file(training_csv, "results", "training_history.csv")
    history = pd.read_csv(training_csv)
    for column in history.columns:
        if column.strip().startswith(("train/", "val/", "metrics/")):
            for epoch, value in enumerate(history[column].values):
                if pd.notna(value):
                    logger.report_scalar("Training history", column.strip(), float(value), epoch)


Transferred 822/822 items from pretrained weights
Ultralytics 8.4.191 🚀 Python-3.12.15 torch-2.8.0 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data={'train': {'yolo_data': ['/home/sagemaker-user/SageMaker_VisDrone_OVD_2026/runs/yoloe_26/20261028T153524Z_5dd65a8e/base_only.yaml']}, 'val': {'yolo_data': ['/home/sagemaker-user/SageMaker_VisDrone_OVD_2026/runs/yoloe_26/20261028T153524Z_5dd65a8e/base_only.yaml']}}, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hs

2026/10/28 15:36:53 INFO mlflow.tracking.fluent: Experiment with name '/home/sagemaker-user/SageMaker_VisDrone_OVD_2026/runs/yoloe_26/20261028T153524Z_5dd65a8e/training' does not exist. Creating a new experiment.
2026/10/28 15:36:53 INFO mlflow.bedrock: Enabled auto-tracing for Bedrock. Note that MLflow can only trace boto3 service clients that are created after this call. If you have already created one, please recreate the client by calling `boto3.client`.
2026/10/28 15:36:53 INFO mlflow.tracking.fluent: Autologging successfully enabled for boto3.
2026/10/28 15:36:53 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


MLflow: logging run_id(9fbbc8c3a01b4d5e9aa691d94ec41b52) to sqlite:////home/sagemaker-user/mlflow.db
MLflow: disable with 'yolo settings mlflow=False'
Using 9680 train, 119 val images for fraction=1.0 at imgsz=1280
Using 4 dataloader workers
Logging results to /home/sagemaker-user/SageMaker_VisDrone_OVD_2026/runs/yoloe_26/20261028T153524Z_5dd65a8e/training/base_safe
Starting training for 80 epochs...

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
       1/80      13.6G      1.694      1.736   0.002947        191       1280: 100% ━━━━━━━━━━━━ 1210/1210 1.7it/s 11:350.6ss
Validate using the text prompt.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 1.0s/it 8.2s0.6ss
                   all        119       4589      0.494      0.445      0.436       0.26

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size


/opt/conda/lib/python3.12/site-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/80      11.7G      1.731      1.676   0.002777        238       1280: 2% ──────────── 32/1210 1.8it/s 18.4s<10:56

## 5 · Evaluate one whole-image reference and five sliced test overlaps

The checkpoints, prompts, evaluation images, `640 × 640` tile size, extraction threshold, per-tile NMS and global NMS are frozen. For each checkpoint, whole-image inference is run once and sliced inference is run at test overlaps of `10%`, `20%`, `30%`, `40%`, and `50%`. Training overlap remains fixed at `20%` throughout.


In [ ]:
def sliding_positions(full, tile, overlap):
    if full <= tile:
        return [0]
    stride = max(1, round(tile * (1 - overlap)))
    points = list(range(0, full - tile + 1, stride))
    if points[-1] != full - tile:
        points.append(full - tile)
    return sorted(set(points))

def bbox_iou_xyxy(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0., x2 - x1) * max(0., y2 - y1)
    area_a = max(0., a[2] - a[0]) * max(0., a[3] - a[1])
    area_b = max(0., b[2] - b[0]) * max(0., b[3] - b[1])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.

def classwise_nms(predictions):
    chosen = []
    by_class = defaultdict(list)
    for p in predictions:
        by_class[p["category_id"]].append(p)
    for group in by_class.values():
        for p in sorted(group, key=lambda r: -r["score"]):
            if all(bbox_iou_xyxy(p["xyxy"], kept["xyxy"]) < CFG.global_nms_iou for kept in chosen
                   if kept["category_id"] == p["category_id"]):
                chosen.append(p)
    return sorted(chosen, key=lambda r: -r["score"])[:CFG.max_detections_per_image]

def predict_region(model, rgb_array, off_x=0, off_y=0):
    # Array from PIL is RGB; Ultralytics ndarray inputs are interpreted as BGR.
    bgr = np.ascontiguousarray(rgb_array[:, :, ::-1])
    result = model.predict(source=bgr, imgsz=CFG.yolo_image_size, conf=CFG.yolo_extraction_threshold,
                           iou=CFG.yolo_local_nms_iou, max_det=CFG.max_detections_per_image,
                           device=0, verbose=False)[0]
    boxes = result.boxes
    if boxes is None or len(boxes) == 0:
        return []
    coords = boxes.xyxy.detach().cpu().numpy()
    classes = boxes.cls.detach().cpu().numpy().astype(int)
    scores = boxes.conf.detach().cpu().numpy()
    predictions = []
    for b, cls, conf in zip(coords, classes, scores):
        if cls >= len(CATEGORY_IDS):
            continue  # Optional blank background class: never evaluate as an object.
        predictions.append({"category_id": CATEGORY_IDS[cls], "score": float(conf),
                            "xyxy": [float(b[0]+off_x), float(b[1]+off_y),
                                     float(b[2]+off_x), float(b[3]+off_y)]})
    return predictions

def run_predict(model, rgb_array, mode, test_overlap=None):
    H, W = rgb_array.shape[:2]
    if mode == "whole":
        return predict_region(model, rgb_array), 1
    if mode != "sliced" or test_overlap is None:
        raise ValueError("Sliced inference requires one explicit test overlap.")

    xs = sliding_positions(W, CFG.slice_width, test_overlap)
    ys = sliding_positions(H, CFG.slice_height, test_overlap)
    raw = []
    if CFG.include_whole_prediction_in_sliced:
        raw.extend(predict_region(model, rgb_array))
    for y in ys:
        for x in xs:
            tile = rgb_array[y:min(y + CFG.slice_height, H), x:min(x + CFG.slice_width, W)]
            raw.extend(predict_region(model, tile, x, y))
    return classwise_nms(raw), len(xs) * len(ys)


MODELS = {"zero_shot": baseline, "fine_tuned": finetuned}
CONDITIONS = []
for stage in MODELS:
    CONDITIONS.append({
        "name": f"{stage}__whole",
        "stage": stage,
        "mode": "whole",
        "test_overlap": None,
    })
    for overlap in CFG.test_overlap_ratios:
        overlap_percent = int(round(100 * overlap))
        CONDITIONS.append({
            "name": f"{stage}__sliced_overlap_{overlap_percent:02d}",
            "stage": stage,
            "mode": "sliced",
            "test_overlap": float(overlap),
        })

if len({item["name"] for item in CONDITIONS}) != len(CONDITIONS):
    raise RuntimeError("Duplicate evaluation condition names were generated.")

raw_predictions = {}
latencies = {}
for condition_spec in CONDITIONS:
    name = condition_spec["name"]
    stage = condition_spec["stage"]
    mode = condition_spec["mode"]
    test_overlap = condition_spec["test_overlap"]
    model = MODELS[stage]

    # First run an identical warmup image, uncounted, before measuring this condition.
    with Image.open(local_file(eval_selection[0]["image_relpath"])) as first:
        warmup_image = np.asarray(first.convert("RGB"))
    with torch.inference_mode():
        run_predict(model, warmup_image, mode, test_overlap)

    outputs, timings = {}, []
    for row in tqdm(eval_selection, desc=f"Inference {name}"):
        with Image.open(local_file(row["image_relpath"])) as image:
            rgb = np.asarray(image.convert("RGB"))
        torch.cuda.synchronize()
        start = time.perf_counter()
        with torch.inference_mode():
            predictions, n_tiles = run_predict(model, rgb, mode, test_overlap)
        torch.cuda.synchronize()
        millis = 1000 * (time.perf_counter() - start)
        outputs[row["image_relpath"]] = predictions
        timings.append({
            "image_relpath": row["image_relpath"],
            "pipeline_ms": millis,
            "tiles": n_tiles,
            "test_overlap": "whole" if test_overlap is None else f"{test_overlap:.2f}",
        })
    raw_predictions[name] = outputs
    latencies[name] = timings
    print(
        name,
        "mean latency ms:", round(float(np.mean([x["pipeline_ms"] for x in timings])), 2),
        "| mean tiles:", round(float(np.mean([x["tiles"] for x in timings])), 2),
    )


## 6 · COCO AP@[.50:.95], AP50, AR500, per-class and robustness analysis

COCO AP is threshold-swept across all predictions >= `yolo_extraction_threshold`, not Precision at one operating threshold. Operational Precision/Recall/F1 and size/occlusion recall below use `operating_threshold_yolo=0.25` and IoU=0.5. The **custom** bins are tiny: area <16², small: 16²–32², medium: 32²–96², large: ≥96² pixels, using original-image GT areas. The official COCO small bucket includes tiny; these custom bins are disjoint and are labelled accordingly. Evaluate the same GT image list and prompts in all four conditions.


In [ ]:
from contextlib import redirect_stdout
from io import StringIO

def size_bin(area):
    if area < 16**2:
        return "tiny"
    if area < 32**2:
        return "small"
    if area < 96**2:
        return "medium"
    return "large"

gt_records = {}
coco_images = []
coco_anns = []
for img_id, row in enumerate(eval_selection, 1):
    path = local_file(row["image_relpath"])
    with Image.open(path) as image:
        W, H = image.size
    coco_images.append({"id": img_id, "width": W, "height": H, "file_name": row["image_relpath"]})
    gt_records[row["image_relpath"]] = []
    for ann in annotations_for_row(row):
        if ann["category_id"] not in CATEGORY_IDS:
            continue
        x, y, w, h = ann["bbox"]
        x1, y1, x2, y2 = max(0., x), max(0., y), min(float(W), x+w), min(float(H), y+h)
        if x2 <= x1 or y2 <= y1:
            continue
        ground = {"id": len(coco_anns)+1, "image_id": img_id, "category_id": ann["category_id"],
                  "bbox": [x1, y1, x2-x1, y2-y1], "xyxy": [x1, y1, x2, y2],
                  "area": (x2-x1)*(y2-y1), "size_bin": size_bin((x2-x1)*(y2-y1)),
                  "occlusion": ann["occlusion"]}
        gt_records[row["image_relpath"]].append(ground)
        coco_anns.append({key: ground[key] for key in ("id", "image_id", "category_id", "bbox", "area")}
                         | {"iscrowd": 0})
assert coco_anns, "No evaluation ground truth."
coco_gt = COCO()
coco_gt.dataset = {"info": {"description": "VisDrone original fixed-split evaluation"},
                   "licenses": [], "images": coco_images,
                   "categories": [{"id": cid, "name": name} for cid, name in zip(CATEGORY_IDS, CATEGORY_NAMES)],
                   "annotations": coco_anns}
coco_gt.createIndex()

def valid_mean(values):
    valid = np.asarray(values)[np.asarray(values) > -1]
    return float(valid.mean()) if valid.size else None

def measure_coco(outputs):
    predictions = []
    for img_id, row in enumerate(eval_selection, 1):
        for p in outputs[row["image_relpath"]]:
            x1, y1, x2, y2 = p["xyxy"]
            predictions.append({"image_id": img_id, "category_id": p["category_id"],
                                "bbox": [x1, y1, max(0., x2-x1), max(0., y2-y1)], "score": p["score"]})
    if predictions:
        with redirect_stdout(StringIO()):
            coco_dt = coco_gt.loadRes(predictions)
    else:
        coco_dt = COCO()
        coco_dt.dataset = {"images": coco_images, "categories": coco_gt.dataset["categories"],
                           "annotations": [], "info": {}}
        coco_dt.createIndex()
    ev = COCOeval(coco_gt, coco_dt, "bbox")
    ev.params.catIds = CATEGORY_IDS
    ev.params.imgIds = [x["id"] for x in coco_images]
    ev.params.maxDets = [1, 10, CFG.max_detections_per_image]
    ev.params.areaRng = [[0, 1e12], [0, 16**2], [16**2, 32**2], [32**2, 96**2], [96**2, 1e12]]
    ev.params.areaRngLbl = ["all", "tiny", "small", "medium", "large"]
    with redirect_stdout(StringIO()):
        ev.evaluate()
        ev.accumulate()
    precision = ev.eval["precision"]  # IoU, recall step, class, area, maxDets
    recall = ev.eval["recall"]        # IoU, class, area, maxDets
    gt_counts = Counter(g["category_id"] for arr in gt_records.values() for g in arr)
    metrics = {}
    groups = {"all": CATEGORY_IDS, "base": sorted(BASE_IDS), "novel": sorted(NOVEL_IDS)}
    for group, ids in groups.items():
        indexes = [CATEGORY_IDS.index(cid) for cid in ids if gt_counts[cid] > 0]
        metrics[f"{group}_AP"] = valid_mean(precision[:, :, indexes, 0, -1])
        metrics[f"{group}_AP50"] = valid_mean(precision[0, :, indexes, 0, -1])
        metrics[f"{group}_AR500"] = valid_mean(recall[:, indexes, 0, -1])
    per_class = []
    for j, (cid, label) in enumerate(zip(CATEGORY_IDS, CATEGORY_NAMES)):
        per_class.append({"category_id": cid, "category": label, "split_group": "base" if cid in BASE_IDS else "novel",
                          "gt_count": gt_counts[cid], "AP": valid_mean(precision[:, :, j, 0, -1]),
                          "AP50": valid_mean(precision[0, :, j, 0, -1]),
                          "AR500": valid_mean(recall[:, j, 0, -1])})
    sizes = []
    for idx, bin_name in enumerate(("tiny", "small", "medium", "large"), 1):
        sizes.append({"size_bin": bin_name,
                      "gt_count": sum(g["size_bin"] == bin_name for arr in gt_records.values() for g in arr),
                      "AP": valid_mean(precision[:, :, :, idx, -1]),
                      "AP50": valid_mean(precision[0, :, :, idx, -1]),
                      "AR500": valid_mean(recall[:, :, idx, -1])})
    return metrics, per_class, sizes, predictions

def match_operating(outputs):
    counts = Counter()
    attributes = defaultdict(lambda: [0, 0])  # [GT count, matched GT count]
    for row in eval_selection:
        key = row["image_relpath"]
        gts = gt_records[key]
        used = set()
        for g in gts:
            attributes[("size_bin", g["size_bin"])][0] += 1
            occ = {0: "none", 1: "partial", 2: "heavy"}.get(g["occlusion"], f"other_{g['occlusion']}")
            attributes[("occlusion", occ)][0] += 1
        for pred in sorted((p for p in outputs[key] if p["score"] >= CFG.operating_threshold_yolo),
                           key=lambda p: -p["score"]):
            candidates = [(bbox_iou_xyxy(pred["xyxy"], g["xyxy"]), g)
                          for g in gts if g["category_id"] == pred["category_id"] and g["id"] not in used]
            if candidates and max(candidates, key=lambda x: x[0])[0] >= CFG.evaluation_iou:
                g = max(candidates, key=lambda x: x[0])[1]
                used.add(g["id"])
                counts["TP"] += 1
                attributes[("size_bin", g["size_bin"])][1] += 1
                occ = {0: "none", 1: "partial", 2: "heavy"}.get(g["occlusion"], f"other_{g['occlusion']}")
                attributes[("occlusion", occ)][1] += 1
            else:
                counts["FP"] += 1
        counts["FN"] += len(gts) - len(used)
    tp, fp, fn = (counts[x] for x in ("TP", "FP", "FN"))
    P = tp / (tp + fp) if tp + fp else 0.
    R = tp / (tp + fn) if tp + fn else 0.
    F1 = 2*P*R/(P+R) if P+R else 0.
    return {"TP": tp, "FP": fp, "FN": fn, "Precision@0.5": P, "Recall@0.5": R, "F1@0.5": F1}, [
        {"attribute": attr, "level": level, "gt_count": num, "Recall@0.5": hit/num if num else None}
        for (attr, level), (num, hit) in sorted(attributes.items())
    ]

summary_rows, class_rows, size_rows, robustness_rows, coco_predictions = [], [], [], [], {}
for condition_spec in CONDITIONS:
    condition = condition_spec["name"]
    stage = condition_spec["stage"]
    mode = condition_spec["mode"]
    test_overlap = condition_spec["test_overlap"]
    metrics, classes, sizes, preds = measure_coco(raw_predictions[condition])
    operating, robustness = match_operating(raw_predictions[condition])
    times = np.array([x["pipeline_ms"] for x in latencies[condition]])
    summary_rows.append({"condition": condition, "stage": stage, "inference_mode": mode,
                         "training_overlap_height": CFG.overlap_height_ratio,
                         "training_overlap_width": CFG.overlap_width_ratio,
                         "test_overlap": "whole" if test_overlap is None else f"{test_overlap:.2f}",
                         "protocol_id": PROTOCOL_ID, "eval_split": CFG.eval_split,
                         "run_mode": CFG.run_mode, "images": len(eval_selection),
                         "threshold": CFG.operating_threshold_yolo, **metrics, **operating,
                         "mean_pipeline_ms": float(times.mean()), "median_pipeline_ms": float(np.median(times)),
                         "p95_pipeline_ms": float(np.percentile(times, 95)),
                         "fps_from_pipeline_mean": 1000/float(times.mean()),
                         "mean_tiles": float(np.mean([x["tiles"] for x in latencies[condition]]))})
    class_rows.extend({"condition": condition, **item} for item in classes)
    size_rows.extend({"condition": condition, **item} for item in sizes)
    robustness_rows.extend({"condition": condition, **item} for item in robustness)
    coco_predictions[condition] = preds

summary_df = pd.DataFrame(summary_rows)
class_df = pd.DataFrame(class_rows)
size_df = pd.DataFrame(size_rows)
robust_df = pd.DataFrame(robustness_rows)
display(summary_df[["condition", "test_overlap", "all_AP", "all_AP50", "base_AP", "novel_AP",
                    "Precision@0.5", "Recall@0.5", "F1@0.5",
                    "mean_pipeline_ms", "mean_tiles"]])


## 7 · Keep **all** results locally; log compact metrics and file inventory to the NEW ClearML project

The SageMaker project's `weights/yoloe_26/<run_id>/` directory contains the original and fine-tuned checkpoints. `results/yoloe_26/<run_id>/` contains full COCO-format predictions, per-image latency, training history, the evaluation ground-truth copy, tables, run configuration and a file inventory. ClearML stores **all aggregate metric tables and scalar series** and a small inventory with absolute local paths and SHA-256 checksums. It does **not** receive thousands of predictions or large checkpoints. Stopping the space preserves the EBS volume; deleting the space may remove local-only results. Back up critical runs before deleting the space.


In [ ]:
# RESULTS_DIR was created in the persistent local project folder at the start.
tables = {"summary.csv": summary_df, "per_class.csv": class_df,
          "size_bins.csv": size_df, "robustness.csv": robust_df}
for filename, frame in tables.items():
    frame.to_csv(RESULTS_DIR / filename, index=False)

def write_json(filename, value):
    destination = RESULTS_DIR / filename
    destination.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + "\n",
                           encoding="utf-8")
    return destination

write_json("coco_ground_truth.json", coco_gt.dataset)
write_json("config_and_protocol.json", {"config": asdict(CFG), "run_id": RUN_ID,
          "clearml_task_id": task.id, "protocol_dataset_id": PROTOCOL_ID,
          "protocol_fingerprint": FINGERPRINT, "category_ids": CATEGORY_IDS,
          "category_names": CATEGORY_NAMES, "prompts": PROMPTS,
          "base_names": BASE_NAMES, "novel_names": NOVEL_NAMES,
          "eval_image_relpaths": [r["image_relpath"] for r in eval_selection],
          "training_image_relpaths": [r["image_relpath"] for r in train_selection],
          "early_stopping_image_relpaths": [r["image_relpath"] for r in val_selection],
          "platform": platform.platform(), "torch": torch.__version__,
          "ultralytics": ULTRALYTICS_VERSION, "gpu": torch.cuda.get_device_name(0)})
for condition, records in coco_predictions.items():
    write_json(f"predictions_{condition}.json", records)
    write_json(f"latency_{condition}.json", latencies[condition])

for path in sorted(RESULTS_DIR.iterdir()):
    if path.is_file():
        LOCAL_INDEX["results"][path.name] = record_local_file(path)

def log_table(title, frame):
    logger.report_table(title=title, series=CFG.eval_split, iteration=0, table_plot=frame)

for name, frame in (("Test-overlap sweep summary", summary_df), ("Per-class AP", class_df),
                    ("Custom size bins", size_df), ("Occlusion / size operating recall", robust_df)):
    log_table(name, frame)
for _, row in summary_df.iterrows():
    for metric in ("all_AP", "all_AP50", "base_AP", "novel_AP", "base_AP50", "novel_AP50",
                   "all_AR500", "Precision@0.5", "Recall@0.5", "F1@0.5", "mean_pipeline_ms", "mean_tiles"):
        if pd.notna(row[metric]):
            logger.report_scalar(title=metric, series=row["condition"], value=float(row[metric]), iteration=0)

# A small local inventory is uploaded to ClearML so saved outputs are discoverable
# without uploading the model checkpoints or prediction dumps themselves.
inventory = {"run_id": RUN_ID, "run_mode": CFG.run_mode,
             "protocol_dataset_id": PROTOCOL_ID, "protocol_fingerprint": FINGERPRINT,
             "clearml_task_id": task.id,
             "local_weights_dir": str(WEIGHTS_DIR),
             "local_results_dir": str(RESULTS_DIR),
             "inventory_path": str(RESULTS_DIR / "local_inventory.json"),
             "objects": LOCAL_INDEX, "all_metrics": summary_df.to_dict(orient="records")}
manifest_file = RESULTS_DIR / "local_inventory.json"
manifest_file.write_text(json.dumps(inventory, indent=2, allow_nan=False) + "\n", encoding="utf-8")
if manifest_file.stat().st_size >= 2 * 1024 * 1024:
    raise RuntimeError("ClearML inventory unexpectedly exceeds 2 MiB; inspect before uploading.")
task.upload_artifact(name="local_inventory_small", artifact_object=str(manifest_file))
logger.report_text(f"All complete results saved locally: {RESULTS_DIR}")
logger.report_text(f"Protocol Dataset ID: {PROTOCOL_ID} | SHA256 fingerprint: {FINGERPRINT}")
logger.report_text("Model: YOLOE-26s (zero-shot yoloe-26s-seg.pt boxes only; fine-tuned yoloe-26s.yaml + YOLOETrainerFromScratch from released weights).")
logger.report_text("Metric note: training overlap is fixed at 0.20; sliced test overlap is swept from "
                   "0.10 to 0.50. AP uses IoU .50:.95; operating P/R/F1 use conf 0.25, IoU 0.5. "
                   "COCO ignores IDs 0 and 11 without masking detections inside ignored regions.")
logger.flush(wait=True)
task.close()
print("DONE. ClearML project:", CFG.clearml_project)
print("ClearML task:", task.id)
print("Local weights:", WEIGHTS_DIR)
print("Local results:", RESULTS_DIR)
print("Free space (GiB):", round(shutil.disk_usage(ROOT).free / 1024**3, 2))
print("IMPORTANT:", "SMOKE ONLY — do not use for paper" if CFG.run_mode == "smoke" else "Full split evaluated")
